**Dependencies**

In [1]:
!pip install -q diffusers transformers accelerate safetensors

**Imports**

In [2]:
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


**Reproducability**

In [3]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

**Load captions**

In [4]:

LORA_PATH = "/content/lora_16_3000_weights.safetensors"


CSV_PATH = "/content/captions.csv"

OUTPUT_DIR = "/content/generated_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

**Prompt template**

In [5]:
import random
import pandas as pd

# Define components
road_type = [
    "narrow street",
    "two-lane road",
    "urban intersection",
    "residential street"
]

density = [
    "empty",
    "moderate traffic",
    "busy traffic"
]

weather = [
    "clear weather",
    "light rain",
    "heavy rain",
    "foggy conditions"
]

lighting = [
    "bright daytime",
    "overcast daylight",
    "sunset lighting"
]

road_elements = [
    "cars parked on the sides, pedestrians walking",
    "traffic lights, crosswalk markings",
    "buildings on both sides, sidewalks",
    "road signs and lane markings"
]

# Prompt template
def generate_prompt():
    return f"Photorealistic dashcam view of a {random.choice(density)} {random.choice(road_type)}, {random.choice(weather)}, {random.choice(lighting)}, {random.choice(road_elements)}, European city, realistic perspective, natural colors, sharp focus"

# Generate dataset (~2500 samples)
NUM_SAMPLES = 500

prompts = [generate_prompt() for _ in range(NUM_SAMPLES)]

df = pd.DataFrame({"caption": prompts})

print("Dataset size:", len(df))
df.head()

Dataset size: 500


,caption
0,Photorealistic dashcam view of a empty residen...
1,Photorealistic dashcam view of a empty urban i...
2,Photorealistic dashcam view of a moderate traf...
3,Photorealistic dashcam view of a moderate traf...
4,Photorealistic dashcam view of a empty residen...


**Load LoRA SD**

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(device)

pipe.load_lora_weights(
    "/content/",
    weight_name="Lora_16_3000_weights.safetensors"
)

pipe.safety_checker = None  # optional (faster)
pipe.enable_attention_slicing()

print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Model loaded!


**Generation loop**

In [ ]:
negative_prompt = (
    "blurry, low quality, distorted, warped buildings, unrealistic, "
    "floating objects, bad perspective, cartoon, painting, anime, illustration, "
    "deformed cars, extra wheels, broken geometry, fisheye, lens distortion, "
    "HDR, overexposed, underexposed, CGI render, 3D render, video game, "
    "watermark, text, signature, logo"
)

results = []

BATCH_SIZE = 4

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch = df.iloc[i:i+BATCH_SIZE]
    prompts = batch["caption"].tolist()

    generators = [
        torch.Generator(device=device).manual_seed(42 + i + j)
        for j in range(len(prompts))
    ]

    images = pipe(
        prompts,
        negative_prompt=[negative_prompt] * len(prompts),
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generators,
        height=512,
        width=512
    ).images

    for j, img in enumerate(images):
        idx = i + j
        filename = f"img_{idx:05d}.png"
        path = os.path.join(OUTPUT_DIR, filename)

        img.save(path)

        results.append({
            "image": filename,
            "caption": prompts[j]
        })

    # 🔥 (optional but recommended)
    pd.DataFrame(results).to_csv(
        os.path.join(OUTPUT_DIR, "progress.csv"),
        index=False
    )

  0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|          | 1/125 [00:26<54:07, 26.19s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

  2%|▏         | 2/125 [00:51<52:04, 25.40s/it]

...

  0%|          | 0/30 [00:00<?, ?it/s]

 12%|█▏        | 15/125 [07:09<53:40, 29.28s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

 13%|█▎        | 16/125 [07:38<53:12, 29.29s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
import shutil

shutil.make_archive("generated_images", "zip", "/content/generated_images")

print("Zipped!")

Zipped!


**CSV**

In [ ]:
out_df = pd.DataFrame(results)

CSV_OUT_PATH = os.path.join(OUTPUT_DIR, "generated_dataset.csv")
out_df.to_csv(CSV_OUT_PATH, index=False)

print("Saved dataset!")
print("Images:", len(out_df))

**Preview**

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample = out_df.sample(5)

for _, row in sample.iterrows():
    img = Image.open(os.path.join(OUTPUT_DIR, row["image"]))
    plt.imshow(img)
    plt.title(row["caption"])
    plt.axis("off")
    plt.show()